In [7]:

from pathlib import Path
import csv
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import numpy as np
from PIL import Image
import yaml, torch, sys
sys.path.insert(0, '../../')
from utils.train_model import set_seed, get_device

In [32]:

with open('../../configs/defaults.yaml') as f:
    config = yaml.safe_load(f)

with open('../../configs/EX_01.yaml') as f:
    config.update(yaml.safe_load(f))

set_seed(config['seed'])
device = get_device()

RAW_DIR  = Path(config['raw_dir'])
GRAD_DIR = Path(config['grad_dir'])
CSV_PATH = Path(config['csv_path'])

data_dir = Path("./EX_01/Raw/")

[device] using cpu


In [33]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1   = nn.Linear(16*5*5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x):
        x = F.avg_pool2d(F.relu(self.conv1(x)), 2)
        x = F.avg_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

def get_loaders(batch_size: int):
    tf = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize((0.1307,), (0.3081,))])
    train = datasets.MNIST(data_dir, train=True,  download=True, transform=tf)
    test  = datasets.MNIST(data_dir, train=False, download=True, transform=tf)
    return (DataLoader(train, batch_size=batch_size, shuffle=True),
            DataLoader(test,  batch_size=batch_size, shuffle=False))

In [36]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        criterion(model(imgs), labels).backward()
        optimizer.step()


def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

In [40]:
epochs = 10
lr = config['lr']
seed = config['seed']
batch_size = config['batch_size']
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

train_loader, test_loader = get_loaders(batch_size)
model     = LeNet5().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, epochs + 1):
    train(model, train_loader, optimizer, criterion, device)
    acc = evaluate(model, test_loader, device)
    print(f"epoch {epoch}/{epochs} | test acc: {acc:.4f}")

# save
out = Path("Model/MNIST/weights/lenet5_mnist.pt")
out.parent.mkdir(exist_ok=True)
torch.save(model.state_dict(), out)
print(f"saved → {out}")


device: cpu
epoch 1/10 | test acc: 0.9721
epoch 2/10 | test acc: 0.9846
epoch 3/10 | test acc: 0.9853
epoch 4/10 | test acc: 0.9869
epoch 5/10 | test acc: 0.9884
epoch 6/10 | test acc: 0.9865
epoch 7/10 | test acc: 0.9889
epoch 8/10 | test acc: 0.9890
epoch 9/10 | test acc: 0.9894
epoch 10/10 | test acc: 0.9902
saved → Model/MNIST/weights/lenet5_mnist.pt


# Gradients

In [34]:

def make_gradient(size: int) -> np.ndarray:
    c1 = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.float32)
    c2 = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.float32)
    direction = random.choice(['h', 'v', 'd'])
    t = np.linspace(0, 1, size)
    if direction == 'h':
        weights = np.tile(t, (size, 1))
    elif direction == 'v':
        weights = np.tile(t[:, None], (1, size))
    else:
        xx, yy = np.meshgrid(t, t)
        weights = (xx + yy) / 2
    return ((1 - weights[..., None]) * c1 + weights[..., None] * c2).astype(np.uint8)


class MNISTGradientDataset(Dataset):
    def __init__(self, csv_path=CSV_PATH, mnist_root=data_dir, transform=None):
        self.rows  = list(csv.DictReader(open(csv_path)))
        self.mnist = datasets.MNIST(mnist_root, train=True, download=False,
                                    transform=transforms.ToTensor())
        self.tf    = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row      = self.rows[idx]
        grad     = self.tf(Image.open(row['gradient_path']).convert('RGB'))
        digit, _ = self.mnist[int(row['mnist_idx'])]
        return grad, digit, int(row['label'])




In [35]:
GRAD_DIR.mkdir(parents=True, exist_ok=True)

mnist = datasets.MNIST(data_dir, train=True, download=True,
                       transform=transforms.ToTensor())

# generate gradients
paths = []
for i in range(config['num_backgrounds']):
    p = GRAD_DIR / f'grad_{i:05d}.png'
    Image.fromarray(make_gradient(IMG_SIZE)).save(p)
    paths.append(str(p))

with open(CSV_PATH, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['gradient_path', 'mnist_idx', 'label'])
    for path in paths:
        for idx in range(len(mnist)):
            _, label = mnist[idx]
            w.writerow([path, idx, label])

print(f'done. pairings -> {CSV_PATH}')



Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 108MB/s]


Extracting EX_01/Raw/MNIST/raw/train-images-idx3-ubyte.gz to EX_01/Raw/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 5.31MB/s]

Extracting EX_01/Raw/MNIST/raw/train-labels-idx1-ubyte.gz to EX_01/Raw/MNIST/raw



Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 8.12MB/s]


Extracting EX_01/Raw/MNIST/raw/t10k-images-idx3-ubyte.gz to EX_01/Raw/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 4.24MB/s]


Extracting EX_01/Raw/MNIST/raw/t10k-labels-idx1-ubyte.gz to EX_01/Raw/MNIST/raw

done. pairings -> Raw/pairings.csv
